> **Solutions for `1-statistics-fundamentals-lab.ipynb`.** Work through the lab on your own first. Open this notebook to check your answers after you have made a real attempt, or when you have been stuck on a part for a while. Many questions have more than one good answer, so compare your reasoning, not just your wording.

# Lab 1: Statistics Fundamentals

> **Course:** Data Science Foundations
> **Lesson:** Day 1 Lab, companion to `1-statistics-fundamentals.md`
> **Dataset:** `mpg` (seaborn). 398 cars from 1970 to 1982, with fuel efficiency, engine size, weight, and origin
> **Estimated time:** 90 minutes

## Lab Objectives

- Implement mean, median, variance, and standard deviation from scratch, and explain the `ddof` difference between NumPy and pandas
- Decide how to handle missing values and measure how that decision changes your statistics
- Build a descriptive summary table and use it to spot skewness and compare spread across scales
- Detect outliers with Tukey's fences and judge whether they are errors or real observations
- Measure relationships with covariance, Pearson, and **Spearman** correlation, and check them within subgroups
- Show by simulation that the spread of sample means follows the **standard error** formula $\sigma / \sqrt{n}$

**How to work through this lab**

- Cells marked `# TODO` are yours to complete. Replace each `None` or comment with working code.
- Run every **Checkpoint** cell. If it prints `passed`, move on. If it raises an error, fix your code first.
- Each **Question** needs a short written answer in the "Your answer" cell below it.
- The dataset is different from the lesson on purpose. Adapt the lesson code; don't paste it.
- Worked solutions are in the `solutions/` folder. Attempt every part yourself before you open them: the struggle is where the learning happens.

## Part 0: Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

sns.set_theme(style="whitegrid")
rng = np.random.default_rng(42)

mpg = sns.load_dataset("mpg")
print(f"Shape: {mpg.shape}")
mpg.head()

## Part 1: Statistics by Hand

Libraries hide the arithmetic. Write your own versions of the core measures using only plain Python (`sum`, `len`, `sorted`, loops). **No NumPy or pandas in these functions.**

In [ ]:
def my_mean(values):
    return sum(values) / len(values)


def my_median(values):
    s = sorted(values)
    n = len(s)
    mid = n // 2
    if n % 2 == 1:
        return s[mid]
    return (s[mid - 1] + s[mid]) / 2


def my_variance(values, ddof=0):
    m = my_mean(values)
    return sum((x - m) ** 2 for x in values) / (len(values) - ddof)


def my_std(values, ddof=0):
    return my_variance(values, ddof) ** 0.5

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
# These are the worked examples from the lesson notes
assert my_mean([55, 60, 70, 80, 85]) == 70
assert my_median([55, 60, 70, 80, 85]) == 70
assert my_median([55, 60, 70, 80]) == 65
assert abs(my_variance([60, 70, 80]) - 66.667) < 0.001
assert abs(my_std([60, 70, 80]) - 8.165) < 0.001
assert abs(my_variance([60, 70, 80], ddof=1) - 100) < 1e-9
print("Part 1 checks passed")

### 1.2 Why do NumPy and pandas disagree?

Compute the standard deviation of the `mpg` column three ways and compare.

In [ ]:
mpg_values = mpg["mpg"]

std_numpy = np.std(mpg_values)                 # ddof=0
std_pandas = mpg_values.std()                  # ddof=1
std_mine = my_std(list(mpg_values), ddof=1)

print(f"NumPy default : {std_numpy}")
print(f"pandas default: {std_pandas}")
print(f"my_std        : {std_mine}")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(std_mine - std_pandas) < 1e-9
assert std_numpy < std_pandas
print("passed")

**Question 1.** Why do the NumPy and pandas values differ? Which one should you report for this dataset, and when would the difference actually matter?

**Model answer:**

NumPy defaults to `ddof=0` (divide by $n$, the population formula) and pandas defaults to `ddof=1` (divide by $n-1$, the sample formula). These 398 cars are a *sample* of all cars, so the sample version (`ddof=1`) is the right one to report. The two differ by a factor of $\sqrt{398/397}$, which is negligible here. With small samples it matters: at $n = 5$ the population formula underestimates the standard deviation by about 11%.

## Part 2: Missing Values

The lesson dropped missing rows without much discussion. Here you will measure what that decision costs.

In [ ]:
mpg.isna().sum()

In [ ]:
n_missing = mpg["horsepower"].isna().sum()
missing_cars = mpg.loc[mpg["horsepower"].isna(), ["name", "model_year", "horsepower"]]
print(f"{n_missing} cars have a missing horsepower")
display(missing_cars)

hp_dropped = mpg["horsepower"].dropna()
hp_imputed = mpg["horsepower"].fillna(mpg["horsepower"].median())

comparison = pd.DataFrame({
    "dropped": [hp_dropped.mean(), hp_dropped.std(), len(hp_dropped)],
    "median_imputed": [hp_imputed.mean(), hp_imputed.std(), len(hp_imputed)],
}, index=["mean", "std", "n"]).round(3)
comparison

**Question 2.** Median imputation lowered the standard deviation. Why does that always happen, and when would it become a real problem?

**Model answer:**

Every imputed value sits at the median, close to the centre, so each one adds a deviation near zero while increasing $n$. The spread shrinks and the data looks more certain than it is. With 6 of 398 rows missing (1.5%) the effect is tiny. If 30% of a column were missing, median imputation would badly understate the variance and weaken correlations with other columns. In that case, consider model-based imputation or report results both ways.

For the rest of the lab we drop the incomplete rows:

In [ ]:
cars = mpg.dropna().reset_index(drop=True)
numeric_cols = ["mpg", "displacement", "horsepower", "weight", "acceleration"]
print(f"Working with {len(cars)} complete cars")

## Part 3: Centre and Spread Summary

Build one table that answers "what is typical?" and "how much does it vary?" for every numeric column.

In [ ]:
q1 = cars[numeric_cols].quantile(0.25)
q3 = cars[numeric_cols].quantile(0.75)
summary = pd.DataFrame({
    "mean": cars[numeric_cols].mean(),
    "median": cars[numeric_cols].median(),
    "std": cars[numeric_cols].std(),
    "IQR": q3 - q1,
    "CV_pct": cars[numeric_cols].std() / cars[numeric_cols].mean() * 100,
})
summary.insert(2, "mean_minus_median", summary["mean"] - summary["median"])
summary.round(2)

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert {"mean", "median", "mean_minus_median", "std", "IQR", "CV_pct"} <= set(summary.columns)
assert abs(summary.loc["weight", "CV_pct"] - cars["weight"].std() / cars["weight"].mean() * 100) < 1e-6
print("passed")

In [ ]:
mode_cylinders = cars["cylinders"].mode()[0]
mode_origin = cars["origin"].mode()[0]
origin_share = cars["origin"].value_counts(normalize=True).round(3)

print(f"Most common cylinder count: {mode_cylinders}")
print(f"Most common origin: {mode_origin}")
origin_share

**Question 3.**
(a) Which column has the largest spread in absolute terms, and which has the largest *relative* spread (CV)?
(b) Using `mean_minus_median`, which columns look right-skewed? Confirm one of them with a histogram in the cell below.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(cars["displacement"], bins=30, kde=True, ax=ax, color="steelblue")
ax.axvline(cars["displacement"].mean(), color="red", ls="--", label="mean")
ax.axvline(cars["displacement"].median(), color="black", ls=":", label="median")
ax.set_title("Displacement: mean pulled right of the median")
ax.legend()
plt.show()

**Model answer:**

(a) `weight` has by far the largest standard deviation (about 850 lb), but only because it is measured in the thousands. Relative to its mean, `displacement` varies the most (CV about 54%), followed by `horsepower` (about 37%). `acceleration` is the most stable (CV about 18%).

(b) `displacement` (mean about 194 vs median 151), `horsepower` (104 vs 94) and `weight` (2978 vs 2804) all have mean > median, so they are right-skewed. A few big American V8 engines stretch the right tail. `mpg` is mildly right-skewed. `acceleration` is close to symmetric.

## Part 4: Outliers with Tukey's Fences

Write reusable functions, then use them on several columns.

In [ ]:
def tukey_fences(series, k=1.5):
    q1, q3 = series.quantile([0.25, 0.75])
    iqr = q3 - q1
    return q1 - k * iqr, q3 + k * iqr


def tukey_outliers(df, col, k=1.5):
    lower, upper = tukey_fences(df[col], k)
    return df[(df[col] < lower) | (df[col] > upper)]

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
lo, hi = tukey_fences(pd.Series([1, 2, 3, 4, 100]))
assert (lo, hi) == (-1.0, 7.0)
assert len(tukey_outliers(pd.DataFrame({"x": [1, 2, 3, 4, 100]}), "x")) == 1
print("passed")

In [ ]:
for col in numeric_cols:
    lo, hi = tukey_fences(cars[col])
    print(f"{col:<13s} fences = [{lo:8.1f}, {hi:8.1f}]   outliers = {len(tukey_outliers(cars, col))}")

In [ ]:
hp_outliers = tukey_outliers(cars, "horsepower")[["name", "cylinders", "origin", "horsepower"]].sort_values("horsepower", ascending=False)
display(hp_outliers)

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=cars, x="origin", y="horsepower", ax=ax)
ax.set_title("Horsepower by origin")
plt.show()

**Question 4.** Are the horsepower outliers data errors? Should they be removed before analysis?

**Model answer:**

No. They are all 8-cylinder American cars (Pontiac, Buick, Chrysler and similar) with genuinely large engines. The box plot shows that the USA group has a much wider horsepower range than Europe or Japan. They look extreme only when pooled with smaller cars. Removing them would delete a real segment of the market. Tukey's fences *flag* values for investigation; they do not decide what is wrong. Only remove a value when you can show it is an error, such as an impossible value or a unit mix-up.

## Part 5: Relationships

### 5.1 Covariance by hand

The lesson's worked example (hours studied vs score) says the covariance is "positive" but never computes it. Compute it now with your Part 1 functions.

In [ ]:
hours = [2, 4, 6, 8]
scores = [50, 65, 80, 90]

mh, ms = my_mean(hours), my_mean(scores)
cov_by_hand = sum((h - mh) * (s - ms) for h, s in zip(hours, scores)) / len(hours)
r_by_hand = cov_by_hand / (my_std(hours) * my_std(scores))

print(f"Covariance: {cov_by_hand:.2f}")
print(f"Pearson r : {r_by_hand:.4f}")

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
assert abs(cov_by_hand - 33.75) < 1e-9
assert abs(r_by_hand - np.corrcoef(hours, scores)[0, 1]) < 1e-9
print("passed. Note: np.cov(hours, scores) gives 45.0 because it divides by n - 1. r is unaffected.")

### 5.2 Correlation matrix

In [ ]:
corr = cars[numeric_cols].corr()
fig, ax = plt.subplots(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", vmin=-1, vmax=1, ax=ax)
ax.set_title("mpg dataset: correlation matrix")
plt.show()

### 5.3 Pearson vs Spearman

Pearson's $r$ measures **linear** association. **Spearman's $\rho$** is Pearson's $r$ computed on the *ranks* of the values, so it measures any **monotonic** association (always increasing or always decreasing, even if curved). Run this demo first:

In [ ]:
x = np.linspace(0.1, 5, 200)
demo = pd.DataFrame({"x": x, "exp_x": np.exp(x), "parabola": (x - 2.5) ** 2})
print("y = exp(x):       Pearson =", round(demo["x"].corr(demo["exp_x"]), 3),
      " Spearman =", round(demo["x"].corr(demo["exp_x"], method="spearman"), 3))
print("y = (x - 2.5)^2:  Pearson =", round(demo["x"].corr(demo["parabola"]), 3),
      " Spearman =", round(demo["x"].corr(demo["parabola"], method="spearman"), 3))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.scatterplot(data=cars, x="horsepower", y="mpg", alpha=0.6, ax=ax)
ax.set_title("Horsepower vs mpg: a curved, decreasing relationship")
plt.show()

pearson_hp = cars["horsepower"].corr(cars["mpg"])
spearman_hp = cars["horsepower"].corr(cars["mpg"], method="spearman")
print(f"Pearson : {pearson_hp:.3f}")
print(f"Spearman: {spearman_hp:.3f}")

**Question 5.** Why is Spearman's value larger in magnitude than Pearson's for horsepower vs mpg? What does the parabola demo tell you that neither coefficient does?

**Model answer:**

The relationship is monotonic (more horsepower always means lower mpg) but curved: mpg falls steeply at first, then levels off. Pearson penalises the curvature because it measures fit to a straight line. Spearman only cares about rank order, so it scores the relationship higher (about −0.85 vs −0.78).

The parabola has a perfect relationship, yet both coefficients are near 0 because it is neither linear nor monotonic. No single correlation number replaces looking at the scatter plot.

## Part 6: Group Structure

On Day 1 the penguin species hid inside the aggregate numbers. Check whether `origin` plays the same role here.

In [ ]:
by_origin = cars.groupby("origin")["mpg"].agg(["count", "mean", "median", "std"]).round(2)
display(by_origin)

fig, ax = plt.subplots(figsize=(8, 4))
sns.boxplot(data=cars, x="origin", y="mpg", ax=ax)
ax.set_title("mpg by origin")
plt.show()

overall_r = cars["weight"].corr(cars["mpg"])
within_r = cars.groupby("origin")[["weight", "mpg"]].apply(lambda g: g["weight"].corr(g["mpg"]))
print(f"Overall r(weight, mpg): {overall_r:.3f}")
print("Within each origin:")
print(within_r.round(3))

**Question 6.** A colleague says: *"Japanese cars get better mileage, so buying Japanese causes better fuel economy."* Use your results to explain what else could be going on.

**Model answer:**

Japanese and European cars do have higher average mpg than American cars. But the weight-mpg correlation is negative overall (−0.83) *and* within every origin (−0.51 to −0.85), and the American cars in this dataset are much heavier. Weight (and engine size) is a confounding variable: it differs by origin *and* drives fuel economy. To isolate an "origin effect" you would compare cars of similar weight, for example by grouping into weight bands or with regression later in the course. Correlation by group is not causation.

## Part 7: Sampling Distributions and the Standard Error

The lesson showed that sample means vary *less* than individual values, but it did not say *how much* less. The answer is the **standard error of the mean**:

$$SE = \frac{\sigma}{\sqrt{n}}$$

You will confirm it by simulation, treating the 392 cars as the population.

In [ ]:
population = cars["mpg"].to_numpy()
sigma = population.std()
sample_sizes = [5, 20, 80]
n_sims = 2000

sample_means = {}
for n in sample_sizes:
    sample_means[n] = rng.choice(population, size=(n_sims, n), replace=True).mean(axis=1)

se_table = pd.DataFrame({
    "observed_std_of_means": {n: m.std() for n, m in sample_means.items()},
    "theory_sigma_over_sqrt_n": {n: sigma / np.sqrt(n) for n in sample_sizes},
})
se_table.index.name = "n"
se_table.round(3)

In [ ]:
# Checkpoint: run this cell. If it prints 'passed', move on.
for n in sample_sizes:
    ratio = se_table.loc[n, "observed_std_of_means"] / se_table.loc[n, "theory_sigma_over_sqrt_n"]
    assert 0.9 < ratio < 1.1, f"n={n}: observed and theoretical SE differ too much"
print("passed")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(16, 3.5), sharex=True)
sns.histplot(population, bins=30, ax=axes[0], color="grey")
axes[0].set_title("Population: individual cars")
for ax, n in zip(axes[1:], sample_sizes):
    sns.histplot(sample_means[n], bins=30, kde=True, ax=ax, color="darkorchid")
    ax.set_title(f"Sample means, n = {n}")
for ax in axes:
    ax.axvline(population.mean(), color="red", ls="--")
    ax.set_xlim(8, 47)
plt.tight_layout()
plt.show()

**Question 7.** By what factor does the standard error shrink when $n$ goes from 20 to 80? The population is right-skewed. What shape are the sampling distributions, and which theorem explains that?

**Model answer:**

Multiplying $n$ by 4 divides the standard error by $\sqrt{4} = 2$ (from about 1.74 to 0.87 mpg). To halve your uncertainty you need four times the data. Although individual mpg values are right-skewed, the sample means are roughly bell-shaped, and more so as $n$ grows. That is the **Central Limit Theorem**. Together with $SE = \sigma/\sqrt{n}$, it underpins the confidence intervals and hypothesis tests on Thursday.

## Part 8: Wrap-Up

Write 4 to 5 sentences for a non-technical manager summarising what you learned about fuel economy in this dataset. Include one number for centre, one for spread, one relationship, and one caveat about what this sample can and cannot tell us about cars in general.

**Model answer:**

*Example:* A typical car in this 1970 to 1982 sample does about 23 mpg (median 22.8), and most cars fall between roughly 17 and 29 mpg. Heavier cars use much more fuel: weight and mpg are strongly negatively correlated (r ≈ −0.83), and the pattern holds within American, European and Japanese cars too. Japanese cars average about 30 mpg against 20 for American cars, but much of that gap is explained by American cars being heavier, not by origin itself. These are 392 cars from one era, so they describe that sample well but cannot be generalised to modern cars without new data.

> Submit your completed notebook to the Kanban board under your name by end of day.